### (1) **PR Drag Evolution**

PR‐drag equations for the semi‑major axis $a$ and eccentricity $e$:

$$
\frac{da}{dt}
= -\frac{2a}{3\tau_n(a)}\;\frac{1 + \tfrac{3}{2}e^2}{(1 - e^2)^{3/2}}, \quad
\frac{de}{dt}
= -\frac{e}{\tau_e(a)}\;\frac{1}{\sqrt{1 - e^2}}
$$

where:

$$
\tau_n(a) = \frac{1}{3}\frac{a^2 c}{G m_0 \beta}, \quad
\tau_e(a) = \frac{2}{5}\frac{a^2 c}{G m_0 \beta}
$$

---

### (2) **Diffusion in $x = 1/a$ due to planetary kicks**

Each encounter with the planet perturbs the orbit, represented as a stochastic change in inverse semi-major axis:

$$
x' = x + \delta x, \quad \delta x \sim \mathcal{N}(0, \sigma_x^2)
$$

The diffusion width is estimated as:

$$
\sigma_x = \frac{10 M_p}{M_\star a_p}
$$


---

### (3) **Jacobi Constant Conservation**

To estimate the new eccentricity after a random change in $x$, we conserve the Jacobi constant:

$$
C(a, e) = -\frac{Gm_0(1-\beta)}{2a_2} - n_1\sqrt{Gm_0(1-\beta)a_2(1-e_2^2)}
$$

This is inverted to solve for new `e`.

---

### (4) **Collision Probability**

We define the probability of collision per time step as:

$$
p_{\rm col} = \left(\frac{R_p}{a_p}\right)^2
$$

If a uniform random number $\mathcal{U} \in [0, 1]$ is less than $p_{\rm col}$, the particle is assumed to collide with the planet and be removed.


In [1]:
import numpy as np
from scipy.integrate import solve_ivp
from joblib import Parallel, delayed
import matplotlib.pyplot as plt

In [2]:
# --- Physical Constants ---
# Constants
Degree_To_Rad = np.pi/180.
AU_To_Meter = 1.495978707e11
yr = np.pi*1e7 # 365*24*3600 # yr in [s]
G = 6.6743e-11 # SI units
c_light = 3.e8

m_Sun = 1.98847e30 # solar mass in [kg]
R_Sun = 6.957e8 # solar radius in [m]
L_Sun = 3.828e26 # solar luminosity in [watts]
T_Sun = 5772 # solar temperature in [K]
m_J = 1.898e27 # Jupiter mass in [kg]
R_J = 7.1492e7 # Jupiter radius in [m]
a_J = 7.78479e8 # Jupiter semi-major axis in [m]
m_E = 5.9722e24 # Earth mass in [kg]
R_E = 6.371e6 # Earth radius in [m]


In [3]:
# --- PR drag timescales ---
def tau_n(a, m0, beta):
    return (1/3) * a**2 * c_light / (G * m0 * beta)

def tau_e(a, m0, beta):
    return (2/5) * a**2 * c_light / (G * m0 * beta)

# --- PR drag differential equations ---
def dadt(t, y, m0, beta):    
    a, e = y
    tau_n_val = tau_n(a, m0, beta)
    return [
        -2*a / (3*tau_n_val) * (1 + 1.5*e**2)/(1 - e**2)**(1.5),
        -e / tau_e(a, m0, beta) / np.sqrt(1 - e**2)
    ]


def integrate_PR_drag(a0, e0, m0, beta, dt):
    """
    Apply one-orbit PR drag evolution step for a and e, using orbit-averaged rates.
    """
    if not (0.0 <= e0 < 1.0):
        print ('error', e0, a0/a_planet)
        quit ()

    tau_n_val = tau_n(a0, m0, beta)
    tau_e_val = tau_e(a0, m0, beta)

    da_dt = -2 * a0 / (3 * tau_n_val) * (1 + 1.5 * e0**2) / (1 - e0**2)**1.5
    de_dt = -e0 / tau_e_val / np.sqrt(1 - e0**2)

    delta_a = da_dt * dt
    delta_e = de_dt * dt

    x0 = 1/a0
    delta_x = -delta_a/a0**2
    a1 = 1/(x0+delta_x)
    
    e1 = e0 + delta_e

#     if a1 <= 0 or not np.isfinite(a1) or not np.isfinite(e1):
#         return a0, e0  # safeguard

    return a1, e1

# --- Jacobi Constant ---
def jacobi_constant(a, e, m0, beta):
    term1 = -G*m0*(1-beta)/(2*a)
    term2 = -np.sqrt(G*(m0+M_planet)/a_planet**3) * np.sqrt(G*m0*(1-beta)*a*(1-e**2))
    return term1 + term2
    
# --- Eccentricity update from Δx ---
def update_e_given_a(a_new, C_old, m0, beta):

    term1 = - G*m0*(1-beta)/(2*a_new)
    term2 = C_old - term1

    factor = -np.sqrt(G*(m0+M_planet)/a_planet**3) * np.sqrt(G*m0*(1-beta)*a_new)
    inside = (term2 / factor) ** 2
    
    print ('inside', inside)
    
    e_new = np.sqrt(1.0 - inside)
    
    return e_new


# --- Single particle evolution ---
def evolve_particle(i, a0, e0, m0, beta, M_star, R_star,
                    M_planet, a_planet, R_planet,
                    sigma_x, p_col):

    a_hist, e_hist, t_hist = [], [], []
    a, e, t = a0, e0, 0.0
    fate = 'survived'

    while True:
        # record
        a_hist.append(a)
        e_hist.append(e)
        t_hist.append(t)
        
        # orbital period
        P_dust = 2*np.pi * np.sqrt(a**3 / (G*M_star*(1-beta)))
        
        # PR drag step
        a, e = integrate_PR_drag(a, e, m0, beta, P_dust)
            
        C_old = jacobi_constant(a, e, m0, beta)
        
        # kick in 1/a
        x = 1./a
        delta_x = np.random.normal(0, sigma_x)
        x_new = x + delta_x
        a = 1.0 / x_new
        
        # checks
        if a < 0:
            fate = 'ejection'
            break
        
        e = update_e_given_a(a, C_old, m0, beta)
                
        if a*(1-e) < 4 * R_star:
            fate = 'hit star'
            break

        # planet collision
        if np.random.rand() < p_col:
            fate = 'collided'
            # record final step
            a_hist.append(a)
            e_hist.append(e)
            t_hist.append(t+P_dust)
            break

        t += P_dust

    return np.array(a_hist), np.array(e_hist), np.array(t_hist), fate

In [4]:
# Example
m0 = 1.0 * m_Sun # stellar mass in [kg]
beta = 0.1
M_star = m0
R_star = R_Sun
M_planet = 1.0 * m_J
a_planet = 10. * AU_To_Meter
R_planet = R_J

sigma_x = 10 * M_planet / (M_star * a_planet)
p_col = (R_planet/a_planet)**2

n_particles = 100

# --- Initial conditions --- when comes out of resonance (2:1)
a0 = 2**(2/3) * a_planet
e0 = 0.4812

In [1]:
# one particle test
evolve_particle(1, a0, e0, m0, beta, M_star, R_star, M_planet, a_planet, R_planet, sigma_x, p_col)

In [2]:
# %%time

# # --- Run particles in parallel using THREADING backend ---
# results = Parallel(n_jobs=-1, backend="threading")(
#     delayed(evolve_particle)(
#         i, a0, e0, m0, beta, M_star, R_star, M_planet, a_planet, R_planet, sigma_x, p_col
#     ) for i in range(n_particles)
# )


# # --- Unpack results ---
# a_hist = [r[0] for r in results]
# e_hist = [r[1] for r in results]
# t_hist = [r[2] for r in results]
# fates = np.array([r[3] for r in results])

In [ ]:
len(np.where(fates=='hit star')[0])

In [ ]:
# --- Time series of a(t) and e(t) for a few particles ---
plt.figure(figsize=(12, 5))
for idx in range(min(5, len(a_hist))):
    plt.plot(t_hist[idx]/yr, a_hist[idx]/a_planet, label=f'Particle {idx}')
plt.xlabel('Time [yr]')
plt.ylabel('a (a_p)')
plt.title('Semi-major Axis vs Time (Sample)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
for idx in range(min(5, len(e_hist))):
    plt.plot(t_hist[idx]/yr, e_hist[idx], label=f'Particle {idx}')
plt.xlabel('Time [yr]')
plt.ylabel('Eccentricity')
plt.title('Eccentricity vs Time (Sample)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# --- Trajectories in (a, e) space ---
plt.figure(figsize=(8, 6))
for ah, eh, fate in zip(a_hist, e_hist, fates):
    color = {'collided': 'red', 'hit star': 'orange', 'ejection': 'purple'}.get(fate, 'blue')
    plt.plot(ah/a_planet, eh, color=color, alpha=0.6)
plt.xlabel('Semi-major axis a (a_p)')
plt.ylabel('Eccentricity e')
plt.title('Dust Particle Trajectories in (a, e) Space')
plt.grid(True)
plt.tight_layout()
plt.show()



In [ ]:
# --- Final (a, e) scatter by fate ---
final_a = [traj[-1] for traj in a_hist]
final_e = [traj[-1] for traj in e_hist]

plt.figure(figsize=(7, 6))
for fa, fe, fate in zip(final_a, final_e, fates):
    color = {'collided': 'red', 'hit star': 'orange', 'ejection': 'purple'}.get(fate, 'blue')
    plt.scatter(fa / AU_To_Meter, fe, color=color, edgecolor='k', s=50)
plt.xlabel('Final a (a_p)')
plt.ylabel('Final e')
plt.title('Final (a, e) Distribution by Fate')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
from collections import Counter

# assume `fates` is your array of strings, e.g., np.array(['survived', 'collided', ...])
counts = Counter(fates)
# sort by descending count for nicer display
labels, values = zip(*sorted(counts.items(), key=lambda kv: -kv[1]))
values = np.array(values)
fracs = values / values.sum()

fig, ax = plt.subplots()
bars = ax.bar(labels, values)
ax.set_ylabel('Number of particles')
ax.set_title('Histogram of Particle Fates')
ax.set_xticklabels(labels, rotation=30, ha='right')

# annotate each bar with count and percentage
for i, (v, f) in enumerate(zip(values, fracs)):
    ax.text(i, v + max(values)*0.01, f'{v} ({f:.1%})', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()